In [1]:
# Then import the required modules
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.preprocessing import OrdinalEncoder


df = pd.read_csv('Downloads/Megastore_Dataset_Task_3 3.csv')
print(df)

      OrderID                         ProductName  Quantity      InvoiceDate  \
0      536370         INFLATABLE POLITICAL GLOBE         48   12/1/2010 8:45   
1      536370      SET2 RED RETROSPOT TEA TOWELS         18   12/1/2010 8:45   
2      536370     PANDA AND BUNNIES STICKER SHEET        12   12/1/2010 8:45   
3      536370       RED TOADSTOOL LED NIGHT LIGHT        24   12/1/2010 8:45   
4      536370  VINTAGE HEADS AND TAILS CARD GAME         24   12/1/2010 8:45   
...       ...                                 ...       ...              ...   
8229   580263          DANISH ROSE DELUXE COASTER        12  12/2/2011 12:43   
8230   580263                CACTI TLIGHT CANDLES        16  12/2/2011 12:43   
8231   581316        RED RETROSPOT SUGAR JAM BOWL         1  12/8/2011 11:46   
8232   581316         GLASS  SONGBIRD STORAGE JAR         1  12/8/2011 11:46   
8233   581316            REGENCY SUGAR BOWL GREEN         1  12/8/2011 11:46   

     UnitPrice  TotalCost         Count

In [2]:
# The coding process for both Ordinal and Nominal Variables

# Ordinal encoding
satisfaction_order = {
    "Very Dissatisfied": 1,
    "Dissatisfied": 2,
    "Satisfied": 3,
    "Very Satisfied": 4,
    "Prefer not to answer": 0
}
priority_order = {
    "Medium": 1,
    "High": 2
}

df['CustomerOrderSatisfaction_Ordinal'] = df['CustomerOrderSatisfaction'].map(satisfaction_order)
df['OrderPriority_Ordinal'] = df['OrderPriority'].map(priority_order)

# One-hot encoding for nominal variables
df = pd.get_dummies(df, columns=['PaymentMethod', 'Segment'])

print(df[['CustomerOrderSatisfaction', 'CustomerOrderSatisfaction_Ordinal', 'OrderPriority', 'OrderPriority_Ordinal', 'PaymentMethod_Credit Card', 'PaymentMethod_PayPal', 'Segment_Consumer', 'Segment_Corporate']].head())


  CustomerOrderSatisfaction  CustomerOrderSatisfaction_Ordinal OrderPriority  \
0                 Satisfied                                  3          High   
1                 Satisfied                                  3          High   
2                 Satisfied                                  3          High   
3                 Satisfied                                  3          High   
4                 Satisfied                                  3          High   

   OrderPriority_Ordinal  PaymentMethod_Credit Card  PaymentMethod_PayPal  \
0                      2                       True                 False   
1                      2                       True                 False   
2                      2                       True                 False   
3                      2                       True                 False   
4                      2                       True                 False   

   Segment_Consumer  Segment_Corporate  
0             F

In [3]:
# Columns to drop

drop_these_columns = [  'Quantity', 'InvoiceDate', 'Country', 'DiscountApplied', 'Region', 
                      'ExpeditedShipping']

# These specified columns being dropped

df = df.drop(columns = drop_these_columns)

# Print results
print(df)

      OrderID                         ProductName UnitPrice  TotalCost   \
0      536370         INFLATABLE POLITICAL GLOBE      $0.85     $40.80    
1      536370      SET2 RED RETROSPOT TEA TOWELS      $2.95     $53.10    
2      536370     PANDA AND BUNNIES STICKER SHEET     $0.85     $10.20    
3      536370       RED TOADSTOOL LED NIGHT LIGHT     $1.65     $39.60    
4      536370  VINTAGE HEADS AND TAILS CARD GAME      $1.25     $30.00    
...       ...                                 ...        ...        ...   
8229   580263          DANISH ROSE DELUXE COASTER     $0.85     $10.20    
8230   580263                CACTI TLIGHT CANDLES     $0.42      $6.72    
8231   581316        RED RETROSPOT SUGAR JAM BOWL     $2.55      $2.55    
8232   581316         GLASS  SONGBIRD STORAGE JAR    $12.50     $12.50    
8233   581316            REGENCY SUGAR BOWL GREEN     $4.15      $4.15    

     OrderPriority CustomerOrderSatisfaction  \
0             High                 Satisfied   
1  

In [4]:
# Now grouping the OrderID
df_group = df.groupby('OrderID').any().reset_index()

#Print results
print(df_group)

     OrderID  ProductName  UnitPrice   TotalCost   OrderPriority  \
0     536370         True        True        True           True   
1     536852         True        True        True           True   
2     536974         True        True        True           True   
3     537065         True        True        True           True   
4     537463         True        True        True           True   
..       ...          ...         ...         ...            ...   
436   581001         True        True        True           True   
437   581171         True        True        True           True   
438   581279         True        True        True           True   
439   581316         True        True        True           True   
440   581587         True        True        True           True   

     CustomerOrderSatisfaction  CustomerOrderSatisfaction_Ordinal  \
0                         True                               True   
1                         True               

In [5]:
df_apriori = df_group.drop(columns=['OrderID', 'ProductName', 'OrderPriority_Ordinal',
                                     'CustomerOrderSatisfaction_Ordinal','Segment_Consumer',
                                      'Segment_Corporate', 'PaymentMethod_Credit Card', 'PaymentMethod_PayPal'])

print(df_apriori.head())

   UnitPrice   TotalCost   OrderPriority  CustomerOrderSatisfaction
0        True        True           True                       True
1        True        True           True                       True
2        True        True           True                       True
3        True        True           True                       True
4        True        True           True                       True


In [27]:
df_apriori.to_excel(r'Downloads/Clean_Apriori_Data_Products.xlsx', index = False)

In [28]:
# Analysis for Market Basket ----->


In [33]:
#Apriori Algorthim
freq_items = apriori(df_apriori, min_support = 0.01, use_colnames = True)

#Create the rules for association

rule = association_rules(freq_items, metric = 'lift', min_threshold = 1)

#print results
print(rule.head())

                   antecedents      consequents  antecedent support  \
0                 (TotalCost )     (UnitPrice )                 1.0   
1                 (UnitPrice )     (TotalCost )                 1.0   
2                 (UnitPrice )  (OrderPriority)                 1.0   
3              (OrderPriority)     (UnitPrice )                 1.0   
4  (CustomerOrderSatisfaction)     (UnitPrice )                 1.0   

   consequent support  support  confidence  lift  representativity  leverage  \
0                 1.0      1.0         1.0   1.0               1.0       0.0   
1                 1.0      1.0         1.0   1.0               1.0       0.0   
2                 1.0      1.0         1.0   1.0               1.0       0.0   
3                 1.0      1.0         1.0   1.0               1.0       0.0   
4                 1.0      1.0         1.0   1.0               1.0       0.0   

   conviction  zhangs_metric  jaccard  certainty  kulczynski  
0         inf            0.0 

/opt/anaconda3/lib/python3.13/site-packages/mlxtend/frequent_patterns/association_rules.py:186: RuntimeWarning: invalid value encountered in divide
  cert_metric = np.where(certainty_denom == 0, 0, certainty_num / certainty_denom)


In [34]:
print(rule)

                                          antecedents  \
0                                        (TotalCost )   
1                                        (UnitPrice )   
2                                        (UnitPrice )   
3                                     (OrderPriority)   
4                         (CustomerOrderSatisfaction)   
5                                        (UnitPrice )   
6                                        (TotalCost )   
7                                     (OrderPriority)   
8                                        (TotalCost )   
9                         (CustomerOrderSatisfaction)   
10                        (CustomerOrderSatisfaction)   
11                                    (OrderPriority)   
12                           (TotalCost , UnitPrice )   
13                        (TotalCost , OrderPriority)   
14                        (UnitPrice , OrderPriority)   
15                                       (TotalCost )   
16                             

In [31]:
#Confidence and lift are the two best predictors then sort table by lift

rule_sorted_by_lift = rule.sort_values(by='lift', ascending=False)

rule_sorted_by_lift.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(UnitPrice ),(ProductName),1.0,1.0,1.0,1.0,1.0,1.0,0.0,inf,0.0,1.0,0.0,1.0
113,"(CustomerOrderSatisfaction, ProductName)","(UnitPrice , OrderPriority)",1.0,1.0,1.0,1.0,1.0,1.0,0.0,inf,0.0,1.0,0.0,1.0
115,"(UnitPrice , ProductName)","(CustomerOrderSatisfaction, OrderPriority)",1.0,1.0,1.0,1.0,1.0,1.0,0.0,inf,0.0,1.0,0.0,1.0
116,"(UnitPrice , OrderPriority)","(CustomerOrderSatisfaction, ProductName)",1.0,1.0,1.0,1.0,1.0,1.0,0.0,inf,0.0,1.0,0.0,1.0
117,"(ProductName, OrderPriority)","(CustomerOrderSatisfaction, UnitPrice )",1.0,1.0,1.0,1.0,1.0,1.0,0.0,inf,0.0,1.0,0.0,1.0
